# Universal phase retrieval
Minimal use of `phase_retrieval_universal.py`.

In [ ]:
import numpy as np
from library import phase_retrieval_universal as pr

In [ ]:
holograms = np.load("data/universal_holograms.npy")  # (n_observations, nx, ny)
mask_pixel = np.load("data/mask_pixel.npy")
supportmask = np.load("data/supportmask.npy")
state_labels = ["saturated", "saturated", "domains", "domains"]
energy_labels = [778.0, 778.0, 780.0, 780.0]
polarizations = [+1, -1, +1, -1]
illumination_labels = ["beam1", "beam1", "beam1", "beam1"]
saturated_states = {"saturated": +1}

In [ ]:
recipe = {
    # Phase-retrieval schedule for every observation.
    "inner_mode": ["HAPRE", "ER"],       # Algorithms repeated in every outer cycle.
    "inner_Nit": [700, 50],               # Iterations in each inner stage.
    "outer_iterations": 100,              # Number of update-plus-projection cycles.
    "warmup_mode": ["HAPRE"],            # Independent stages before coupling observations.
    "warmup_Nit": 0,                      # Zero disables warmup.
    "shuffle_observations": True,         # Randomize observation order each cycle.
    "random_seed": None,                  # Seed for update-order randomization.
    "beta_zero": 0.5,                     # Beta value(s), scalar or one per stage.
    "beta_mode": "arctan",               # Beta schedule name(s) or arrays.
    "alpha_zero": 0.0,                    # TV strength; zero disables TV.
    "alpha_mode": "const",               # Alpha schedule name(s) or arrays.
    "TV_freq": 1e9,                       # TV update interval.
    "warmup_beta_zero": None,             # None inherits beta_zero.
    "warmup_beta_mode": None,             # None inherits beta_mode.
    "warmup_alpha_zero": None,            # None inherits alpha_zero.
    "warmup_alpha_mode": None,            # None inherits alpha_mode.
    "warmup_TV_freq": None,               # None inherits TV_freq.
    "plot_every": 1e9,                    # Error sampling/plot interval.
    "average_img": 1,                     # Number of best late iterates to average.
    "Fourier_last": True,                 # Finish each stage with its Fourier constraint.
    "final_fourier_constraint": True,     # Finish final fields on measured amplitudes.
    "hologram_intensity_cutoff_vmin": -1, # Percentile baseline subtraction.
    # Universal object projection.
    "projection_model": "physical_factorized", # physical_factorized, state_energy_beam, none, svd, rank1_spectral.
    "projection_every": 1,                # Projection interval in outer cycles.
    "projection_start": 0,                # First cycle eligible for projection.
    "projection_relaxation": 1.0,         # 0 keeps current objects; 1 applies full projection.
    "projection_constraints_inside_support_only": False, # If True, apply joint projections only inside supportmask.
    "observation_weights": None,          # Positive weight per input hologram.
    "rank_deficient": "error",           # Error or accept a minimum-norm flexible-model fit.
    "log_floor": 1e-12,                   # Magnitude floor before complex logarithm.
    # Physical mixed-data factorization.
    "physical_iterations": 20,            # Alternating-fit iterations per projection.
    "saturated_states": saturated_states, # Optional state-to-+1/-1 mapping.
    "charge_spectral_constraint": "free",   # free, kk, known_beta, known_beta_kk.
    "magnetic_spectral_constraint": "free", # Independent magnetic spectral constraint.
    "energy_values": np.array([778.0, 780.0]), # Unique-energy order; required by KK/index conversion.
    "known_charge_beta_spectrum": None,   # Known charge beta(E), if using refractive-index input.
    "known_charge_delta_spectrum": None,  # Known charge delta(E).
    "known_magnetic_beta_spectrum": None, # Known magnetic beta(E).
    "known_magnetic_delta_spectrum": None,# Known magnetic delta(E).
    "charge_absorption_part": "real",    # Advanced response convention: real or imag.
    "magnetic_absorption_part": "real",  # Advanced response convention: real or imag.
    "charge_response_real_range": None,   # Direct optional bounds on Re(q_charge).
    "charge_response_imag_range": None,   # Direct optional bounds on Im(q_charge).
    "magnetic_response_real_range": None, # Direct optional bounds on Re(q_magnetic).
    "magnetic_response_imag_range": None, # Direct optional bounds on Im(q_magnetic).
    "kk_sign": 1.0,                       # Sign convention multiplying KK dispersion.
    "kk_subtract_baseline": True,         # Remove endpoint baseline before KK.
    "kk_normalize_input": False,          # Normalize absorption before KK.
    "known_spectrum_normalization": "none", # none, maxabs, l2, or std.
    "fit_known_spectrum_scale": True,     # Fit scale of physical known spectra.
    "fit_known_spectrum_offset": True,    # Fit offset of physical known spectra.
    # Pure-energy SVD/rank-one options; used only for pure energy scans.
    "rank": 1,                            # Residual rank for projection_model="svd".
    "projection_static_mode": "mean",    # Static object: mean, first, or none.
    "spectral_constraint": "free",       # Rank-one constraint: free, kk, known_beta, known_beta_kk.
    "known_beta_spectrum": None,          # Known absorption-like rank-one response spectrum.
    "known_delta_spectrum": None,         # Known dispersion-like rank-one response spectrum.
    "absorption_part": "real",           # Rank-one absorption location: real or imag.
    "known_beta_normalization": "none",  # none, maxabs, l2, or std.
    "fit_known_beta_scale": True,         # Fit rank-one known-spectrum scale.
    "fit_known_beta_offset": True,        # Fit rank-one known-spectrum offset.
    # Preferred identifiable physical bounds: q=-ikt*n.
    "charge_kt_delta_range": None,        # Optional bounds on k*t*delta_charge.
    "charge_kt_beta_range": None,         # Optional bounds on k*t*beta_charge.
    "magnetic_kt_delta_range": None,      # Optional bounds on k*t*delta_magnetic.
    "magnetic_kt_beta_range": None,       # Optional bounds on k*t*beta_magnetic.
    # Conversion of supplied delta(E)/beta(E) to dimensionless q(E).
    "wave_numbers": None,                 # k(E) in m^-1; otherwise derived from energy_values.
    "thickness": None,                    # Thickness in metres, required for index conversion.
    "known_charge_kt_beta_spectrum": None,   # Known k*t*beta_charge(E), no thickness needed.
    "known_charge_kt_delta_spectrum": None,  # Known k*t*delta_charge(E).
    "known_magnetic_kt_beta_spectrum": None, # Known k*t*beta_magnetic(E).
    "known_magnetic_kt_delta_spectrum": None,# Known k*t*delta_magnetic(E).
}

fields, fieldswarmup, components, bsmasks, errors = (
    pr.universal_phase_retrieval_algorithm(
        holograms,
        mask_pixel,
        supportmask,
        state_labels=state_labels,
        energy_labels=energy_labels,
        polarization_coefficients=polarizations,
        illumination_labels=illumination_labels,
        saturated_states=saturated_states,
        universal_recipe=recipe,
    )
)